In [1]:
##### IMPORTS #####
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
##### END OF IMPORTS #####

##### USER INPUTS #####

#INPUT: Please define the symbols of your coordinate system here, make your timelike coordinate the FIRST symbol!
coords = sp.symbols('t r theta phi')

for c in coords: #DO NOT TOUCH THIS LOOP (this loop makes the coordinates symbols)
    globals()[str(c)] = c


#INPUT: Please input if this particle is massless or not
massless_particle = False


#INPUT: Please define your metric here, make each column refer to above symbols in the same order!
g = sp.Matrix(
    [[-(1 - (10 / r)) , 0 , 0 , 0] ,                #TIMELIKE COLUMN
     [0 , 1/(1 - (10 / r)) , 0 , 0] ,               #SPACELIKE COLUMN
     [0 , 0 , r**2 , 0] ,                           #SPACELIKE COLUMN
     [0 , 0 , 0 , r**2 * sp.sin(theta)**2]]         #SPACELIKE COLUMN
)


#INPUT: Please input if this metric should be physically viable.
physically_viable = True


#INPUT: Please input your initial 4-position, make each entry refer to coordinates in same order as above!
ri = [0 , 100 , sp.pi/2 , 0]


#INPUT: Please input your initial 3-velocity (ie the spatial velocities, not the timelike one), follow same spatial order as above
vi = [.2 , 0 , 0]


#INPUT: Please input the amount of "integration time" you would like this simulation to go by and the step amount
dlam = .005 #Step size
lamTot = 5000 #Total sim code time

##### END OF USER INPUTS #####


##### FUNCTIONS #####

#This function will calculate the inverse metric and corresponding Christoffel symbols
def calcChristoffel(g , indices):

    #This program will calculate the connection coefficients for g using the Levi-Civita connection

    #Find g inverse
    ginv = g.inv()

    #Initialize the connections as a 4x4x4 array
    christoffel = sp.MutableDenseNDimArray.zeros(4,4,4)

    #The Levi-Civita Connection is
    #Γ^m_np = .5 * g^ms ( deriv_p(g_sn) + deriv_n(g_sp) - deriv_s(g_np) )
    for mu in range(0 , 4):
        for nu in range(0 , 4):
            for rho in range(0 , 4):
                expr = 0
                for sigma in range(0 , 4):
                    expr += .5 * ginv[mu , sigma] * (
                        sp.diff(g[sigma , nu] , indices[rho]) +
                        sp.diff(g[sigma , rho] , indices[nu]) -
                        sp.diff(g[nu , rho] , indices[sigma])
                    )

                christoffel[mu , nu , rho] = expr

    #We have the christoffels yay

    return (christoffel , ginv) #Return ginv also just incase its useful elsewhere

#This function will calculate the
def findUtInit(is_null , ri , vi , g , indices):

    #For our integration to be correct, we need to find the t-component of the initial four velocity.
    #This ut determines the evolution of the system as a massive or massless particle.
    #Per the diff geo, the magnitude of the four velo is found via u^m*u^n*g_mn
    #This magnitude must equal -1 for a massive particle and 0 for a massless particle

    if is_null:
        condition = 0

    else:
        condition = -1

    #Lets sub in the initial position symbols into the metric
    subs_dict = {
        indices[0] : ri[0] ,
        indices[1] : ri[1] ,
        indices[2] : ri[2] ,
        indices[3] : ri[3] ,
    }

    g_initial = g.subs(subs_dict)

    #Lets initialize our unknown ut
    ut = sp.symbols("ut" , positive = True)
    ui = [ut , vi[0] , vi[1] , vi[2]]

    #Now we do the inner product correctly
    expr = 0
    for m in range(0 , 4):
        for n in range(0 , 4):
            expr += ui[m] * ui[n] * g_initial[m , n]

    #Make the expression an equation using condition
    inner_product_constraint = sp.Eq(expr , condition)

    #Solve for ut
    time_part = sp.solve(inner_product_constraint , ut)

    #Put the four velocity together
    initial_4velo = [time_part[0] , vi[0] , vi[1] , vi[2]]

    #Ensure correct magnitude
    expr = 0
    for mu in range(0 , 4):
        for nu in range(0 , 4):
            expr += initial_4velo[mu] * initial_4velo[nu] * g_initial[mu , nu]

    print(f"4-Velocity magnitude is {expr}")
    print(f"4-Velcity is {initial_4velo}")
    return(initial_4velo)

#This function will veify if the given metric is allowed
def verifyMetric(g , ri):

    #A valid spacetime metric is
    #1. A 4x4 matrix
    #2. Symmetric and real
    #3. Has a non-zero determinant
    #4. Has a Lorentzian signature (ie 1 negative eigenvalue and 3 positive eigenvalues since we are using (-,+,+,+))

    #Is this metric symmetric?
    if (not g.is_symmetric()):
        print("Error, this metric is not symmetric. Defaulting to Minkowski spacetime!!!")
        return -1
    assert g.is_symmetric() , "Error, this metric is not symmetric!"

    #Is this metric nonsingular?
    det = g.det()
    assert det != 0 , "Error, this metric is singular!"

    ###Does this metric have the correct (-,+,+,+) signature?
    check_negative = False

    #Lets sub in the initial position symbols into the metric
    subs_dict = {
        indices[0] : ri[0] ,
        indices[1] : ri[1] ,
        indices[2] : ri[2] ,
        indices[3] : ri[3] ,
    }

    g_initial = g.subs(subs_dict)
    eigs = g_initial.eigenvals()

    for (key,value) in eigs.items():

        if key < 0:
            if value == 1:
                check_negative = True

    assert check_negative , "Error, this metric does not have the correct Lorentzian signature (-,+,+,+)!"





#This function will form the geodesic equation
#make it 2 for loops through the implied summations (from 0 to 3) of the geodesic equations, add to an expression variable and return

#This function will integrate the geodesic equation
#rk4

#This function will make 6 parametric plots

##### END OF FUNCTIONS #####



##### RUN SIM #####

#Must check if metric is 4x4 and there are four coordinates
assert len(coords) == 4 , "Error, not enough/too many coordinates!"
assert g.shape == (4,4) , "Error, metric is not 4x4!"

#Create an indices dictionary
indices = {i: coords[i] for i in range(4)}
print(indices)

#If physically viable metrics are set, check if this metric works
if physically_viable:
    verifyMetric(g = g , ri = ri)

#Calculate inverse metric and christoffel symbols
(Connections , ginv) = calcChristoffel(g = g , indices = indices)
print(Connections)

#Calculate the time component of the initial 4-velocity
ui = findUtInit(is_null = massless_particle, ri = ri , vi = vi , g = g , indices = indices)


#Integrate the geodesic equations

#Calculate the initial timelike component of the 4-velocity
#ui = findUtInit(is_null = massless_particle, ri = ri , vi = vi , g = g , indices = indices)

{0: t, 1: r, 2: theta, 3: phi}
[[[0, 5.0/(r*(r - 10)), 0, 0], [5.0/(r*(r - 10)), 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0]], [[5.0*(r - 10)/r**3, 0, 0, 0], [0, -5.0*(r - 10)/(r**3*(1 - 10/r)**2), 0, 0], [0, 0, 10.0 - 1.0*r, 0], [0, 0, 0, -1.0*(r - 10)*sin(theta)**2]], [[0, 0, 0, 0], [0, 0, 1.0/r, 0], [0, 1.0/r, 0, 0], [0, 0, 0, -1.0*sin(theta)*cos(theta)]], [[0, 0, 0, 0], [0, 0, 0, 1.0/r], [0, 0, 0, 1.0*cos(theta)/sin(theta)], [0, 1.0/r, 1.0*cos(theta)/sin(theta), 0]]]
4-Velocity magnitude is -1.00000000000000
4-Velcity is [1.07726219053696, 0.2, 0, 0]
